# 30. The encoder's inner split count

**One variable against ledger row 38** (`xgb_te`, CV 0.967099): the number of inner
folds the target encoder uses to build training-row encodings. Same learner, same outer
folds, same seed, same budget, same smoothing constant. Only `N_INNER` moves.

## The last unvaried constant

`13_target_encoding.ipynb` fixed two constants in cell 1 on 2026-08-11, `SMOOTH = 10.0`
and `N_INNER = 5`, both chosen once and never touched. Every model since inherited both:
the five LightGBM seeds, the five CatBoost seeds, both neural models and the five
XGBoost seeds, which is 22 of the 29 members of row 44. `29_encoder_smooth.ipynb` swept
the first and found it inert, ledger rows 46 to 49. This sweeps the second. After it
there is no constant in this pipeline that has never been varied.

## What this knob actually touches, which is less than it looks

`encode_fold` fits the encoder twice, for two different sets of rows. Validation and
test rows get an encoding fit once on the whole training portion. Training rows get
inner out-of-fold values from a `KFold(N_INNER)` split of that same portion. `N_INNER`
is read by the second of those and by nothing else.

So the validation and test features are **bit-identical across every arm here**. The
knob changes what the model learns from, and never changes what it is scored on.

That makes a sharper positive control available than `29` had. There, `SMOOTH` reached
every encoded column, so the control was one-sided: the columns had to move. Here the
control is two-sided. Training-row encodings **must** differ between arms, and
validation-row encodings **must not**, to the bit. Both are checked in stage 3 before
anything trains. A knob that failed to reach the encoder and a knob that reached too far
are different faults and this run can tell them apart.

## The prior, from the same arithmetic that called `SMOOTH`

`N_INNER` sets how much data each inner fit sees, `(N_INNER - 1) / N_INNER` of the
training portion: 50 percent at 2, 80 percent at 5, 95 percent at 20. Against roughly
500 rows per level on the float columns, the level counts inside an inner fit run from
about 250 to about 475 across that grid, and at `SMOOTH = 10` the pull toward the prior
is 3.8 percent at the low end against 2.1 percent at the high end. On the three
categorical columns, whose levels hold tens of thousands of rows, it is inert twice over.

There is a second effect and it is the one worth watching. Training-row encodings are
built on less data than the validation-row encodings the model is later scored against,
which is a train/serve mismatch, and a larger `N_INNER` narrows it. That argues for a
curve rising gently toward 20 rather than a flat one. It is the only mechanism here
predicting anything other than a null, and it is why the grid runs to 20 instead of
stopping at 10.

## The prediction, written before the run

`NOTES.md` records five reopenings and the rule they support: a change of representation
revalues learners, not their knobs. Three learner changes paid, two hyperparameter
changes returned nothing. `N_INNER` is a hyperparameter of a component already fitted to
this representation, so the rule predicts **null**. This is the sixth point on it.

The mismatch argument above is the honest case against that prediction. Recording both
before the run, because the value of the rule is entirely in whether it was written down
first.

## The gate, pre-registered, and it is `29`'s

Reused verbatim from `29_encoder_smooth.ipynb`, floor included, so the two sweeps are
directly comparable. `23_lgbm_leaves.ipynb` set its floor at the fold spread, 0.00045.
That floor is not used here and the reason is recorded in `29` and in `NOTES.md`: it
would have called XGBoost's +0.000316 a null.

| verdict | condition |
|---|---|
| `carry to a second seed` | wins >= 4/5, mean > 2 x paired sd, and mean >= +0.00010 |
| `parity` | paired-significant but under +0.00010 |
| `null` | anything else |

The +0.00010 floor sits just below +0.000132, CatBoost's margin in row 26, which is the
smallest single-model difference this repo has since confirmed real through a seed
sweep. No arm is called an improvement on one seed, per rows 26 and 38.

## What this costs, which `29` did not have to think about

`SMOOTH` cost the same in every arm. `N_INNER` does not: the inner loop runs `N_INNER`
times per column, so the `20` arm builds its encoder about four times slower than the
`5` arm. The grid sums to 40 inner fits per fold against `29`'s 25, so the encoder bill
is roughly 1.6x `29`'s. Against 2000 XGBoost trees per arm it is still not the cost that
matters, and `29`'s full run took 54m 21s on Kaggle.


In [ ]:
# One flag. The sweep always runs top to bottom on Kaggle.
SMOKE = True

SEED = 42

# The knob under test. 5 is 13's value and therefore row 38's, so that arm is the
# reproduction check. The encoder reads N_INNER from globals, same as 29 did with
# SMOOTH, so an arm is a rebinding rather than a second encoder.
INNERS = [2, 3, 5, 10, 20]
BASE_INNER = 5
N_INNER = BASE_INNER
PROBE_INNER = 20        # the far arm, used for the two-sided liveness check

# Row 38's configuration, held. SMOOTH is 13's value and 29 confirmed it sits on the
# flat part of its own curve, so holding it here holds a defensible point.
SMOOTH = 10.0
LR = 0.05
N_EST = 2000
BENCH_EST = 200
PROBE_FOLD = 0
MAX_DEPTH = 6
N_JOBS = -1

BASELINE_NAME = "xgb_te"
BASELINE_CV = 0.967099
EXPECTED_FOLD_SHA = "ec282b0968059676"

EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

GATE_FLOOR = 1.0e-04

print(f"SMOKE = {SMOKE}   inner splits {INNERS}   base {BASE_INNER}")


## Stage 1. Data, folds, leak checklist

The fold checksum is the only thing standing between an out-of-fold vector that
blends and one that is silently misaligned, so it is checked before anything trains
rather than after.

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

# Runs here or on Kaggle. Both are found by name rather than by assuming a shape.
KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
SUB = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "submissions"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

# Leak checklist, re-run rather than ticked by inspection. `id` is a contiguous row
# index that separates train from test perfectly, so it is a guaranteed leak if it
# ever reaches the model.
checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# ROW_IDX maps this run's rows back into the saved member vectors.
if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    # Measured 2026-08-19 and written up in NOTES.md: at 16,000 rows this machine
    # runs 101x slower at n_jobs=-1 than at n_jobs=1, monotone in the thread count.
    # N_JOBS above is chosen to match row 17 on Kaggle at 691,369 rows, where it is
    # right. A smoke run produces no ledger number, so overriding it here costs
    # nothing and is the difference between two minutes and giving up on the check.
    N_JOBS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

## Stage 2. The encoder

Copied from `13_target_encoding.ipynb` so the feature set is row 17's feature set. A
copy is a provenance risk under the notebook layout, so it is checked rather than
asserted: the cell below parses the encoder out of `13`, normalises both versions
through `ast.unparse`, and compares checksums. Expected fingerprint
`0642e41750ef8bab`, the same value rows 26 and 33 recorded.

In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

### The leak checks, by execution

The same three checks `13` ran, on the same encoder, so their numbers are directly
comparable to the ones in `NOTES.md`. Read all three together: the first two must be
about zero, the third must be large. Without the third, an encoder that ignored the
target entirely would pass the first two and look clean.

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## Stage 3. Bench, determinism, and the two-sided liveness check

`29` needed this stage because a knob that never reached the encoder would have produced
five identical arms and a perfectly flat curve, which is exactly the result its header
predicted. The same trap is here and the header makes the same prediction, so the same
check is run.

It is stronger here because `N_INNER` has a side it must not reach. Two builds of the
same fold at 5 and at 20:

- the **training** encodings must differ, or the arms are five copies
- the **validation** encodings must be identical to the bit, or `N_INNER` is leaking
  into the fit that scores the model and this is not the experiment described above

A failure of the second would be the more interesting one, and nothing before this
notebook would have caught it.


In [ ]:
import xgboost as xgb


def make(n_est):
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True,
        learning_rate=LR, n_estimators=n_est, max_depth=MAX_DEPTH,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=N_JOBS, verbosity=0,
    )


def fit_arm(Xtr, ytr, Xva, n_est, Xte=None):
    m = make(n_est)
    t0 = time.time()
    m.fit(Xtr, ytr)
    secs = time.time() - t0
    p = m.predict_proba(Xva)[:, 1]
    p_te = m.predict_proba(Xte)[:, 1] if Xte is not None else None
    return p, p_te, secs


def hhmm(s):
    return f"{int(s // 60)}m {int(s % 60):02d}s"


LOG = (Path("/kaggle/working") if ON_KAGGLE
       else LOCAL / "artifacts" / "logs") / "30_inner_splits.log"
LOG.parent.mkdir(parents=True, exist_ok=True)


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, inner={INNERS} ===")

tr0 = np.where(folds != PROBE_FOLD)[0]
va0 = np.where(folds == PROBE_FOLD)[0]

# Two-sided liveness. Build the same fold at the base and the far arm and compare the
# encoded columns directly, before any model sees them.
N_INNER = BASE_INNER
_t0 = time.time()
Xa_tr, Xa_va, _ = build(X, y, tr0, va0)
ENC_SECS = time.time() - _t0

N_INNER = PROBE_INNER
Xb_tr, Xb_va, _ = build(X, y, tr0, va0)
N_INNER = BASE_INNER

te_cols = [c for c in Xa_tr.columns if c.startswith("te_")]
moved_tr = float(np.abs(Xa_tr[te_cols].to_numpy()
                        - Xb_tr[te_cols].to_numpy()).max())
moved_va = float(np.abs(Xa_va[te_cols].to_numpy()
                        - Xb_va[te_cols].to_numpy()).max())

INNER_LIVE = moved_tr > 0
VAL_HELD = moved_va == 0.0
print(f"encoder, one fold at N_INNER={BASE_INNER}: {hhmm(ENC_SECS)}   "
      f"{Xa_tr.shape[1]} features")
print(f"N_INNER {BASE_INNER} vs {PROBE_INNER}, largest change in a TRAIN encoding: "
      f"{moved_tr:.3e}  "
      f"{'live' if INNER_LIVE else 'NOT LIVE - the sweep would be five copies'}")
print(f"N_INNER {BASE_INNER} vs {PROBE_INNER}, largest change in a VALID encoding: "
      f"{moved_va:.3e}  "
      f"{'held, as it must be' if VAL_HELD else 'MOVED - N_INNER is reaching the fit that scores the model'}")
del Xb_tr, Xb_va
gc.collect()

# Xa_* was built at BASE_INNER on PROBE_FOLD, so the bench reuses it rather than
# paying for a third build of the same thing.
pa, _, sa = fit_arm(Xa_tr, y[tr0], Xa_va, BENCH_EST)
pb, _, _ = fit_arm(Xa_tr, y[tr0], Xa_va, BENCH_EST)
delta = float(np.abs(pa - pb).max())
DETERMINISTIC = delta == 0.0
print(f"determinism, two identical runs, max |diff|: {delta:.3e}  "
      f"{'OK' if DETERMINISTIC else 'NOT REPRODUCIBLE'}")

# Unlike 29 the encoder bill is not flat across arms, so the projection scales it by
# the arm's inner count. Approximate: part of ENC_SECS is the fit-once pass, which
# does not scale, so this over-projects the wide arms slightly.
per_tree = sa / BENCH_EST
total = 5 * sum(ENC_SECS * (n / BASE_INNER) + per_tree * N_EST for n in INNERS)
print()
print(f"projection, {N_EST} trees, 5 folds x {len(INNERS)} arms: {hhmm(total)}")
note(f"stage 3 done, determinism {'OK' if DETERMINISTIC else 'FAILED'}, "
     f"inner live {INNER_LIVE}, valid held {VAL_HELD}, projected {hhmm(total)}")

del Xa_tr, Xa_va, pa, pb
gc.collect()


## Stage 4. The sweep

Each arm rebinds `N_INNER`, rebuilds the encoder inside every fold, and trains the same
XGBoost configuration. The fold partition, the model seed and the smoothing constant are
identical across arms.


In [ ]:
oof = {n: np.zeros(len(train)) for n in INNERS}
test_pred = {n: np.zeros(len(test)) for n in INNERS}
per_fold = {n: [] for n in INNERS}

t0 = time.time()
for f in range(5):
    tr = np.where(folds != f)[0]
    va = np.where(folds == f)[0]
    for n in INNERS:
        N_INNER = n
        Xtr, Xva, Xte = build(X, y, tr, va, X_test)
        p, p_te, secs = fit_arm(Xtr, y[tr], Xva, N_EST, Xte)
        oof[n][va] = p
        test_pred[n] += p_te / 5
        per_fold[n].append(float(roc_auc_score(y[va], p)))
        note(f"  fold {f} inner {n:>3}: {per_fold[n][-1]:.6f}  ({hhmm(secs)})")
        del Xtr, Xva, Xte
        gc.collect()
    done = time.time() - t0
    note(f"fold {f} done, elapsed {hhmm(done)}, "
         f"about {hhmm(done / (f + 1) * (4 - f))} left")
N_INNER = BASE_INNER

cv = {n: float(np.mean(per_fold[n])) for n in INNERS}
sd = {n: float(np.std(per_fold[n])) for n in INNERS}

print()
print(f"{'inner':>7} {'CV':>10} {'fold sd':>10}")
for n in INNERS:
    star = "  <- 13's value, and row 38's" if n == BASE_INNER else ""
    print(f"{n:>7} {cv[n]:>10.6f} {sd[n]:>10.6f}{star}")
note("sweep done, " + ", ".join(f"{n}:{cv[n]:.6f}" for n in INNERS))


In [ ]:
repro = cv[BASE_INNER] - BASELINE_CV
REPRODUCED = abs(repro) < 1e-4
print(f"arm N_INNER={BASE_INNER}: {cv[BASE_INNER]:.6f}")
print(f"ledger row 38     : {BASELINE_CV:.6f}")
print(f"difference        : {repro:+.2e}   "
      f"{'inside' if REPRODUCED else 'OUTSIDE'} the 1e-04 tolerance")
if SMOKE:
    print("SMOKE: subsampled, so this is EXPECTED to be far out and is not a check.")

base = np.array(per_fold[BASE_INNER])
rows = []
for n in INNERS:
    if n == BASE_INNER:
        continue
    d = np.array(per_fold[n]) - base
    rows.append((n, d.mean(), d.std(ddof=1), int((d > 0).sum()), d))

print()
print(f"{'inner':>7} {'paired mean':>13} {'paired sd':>11} {'wins':>7} {'mean/sd':>9}")
for n, m, sdv, w, _ in rows:
    print(f"{n:>7} {m:>+13.6f} {sdv:>11.6f} {w:>5}/5 "
          f"{(m / sdv if sdv else float('nan')):>9.1f}")
print()
for n, m, sdv, w, d in rows:
    print(f"  {n:>5}: per-fold {np.round(d, 6).tolist()}")

# The mechanism in the header predicts a rising curve. Reported whatever it does,
# because a prediction only counts if the check for it is unconditional.
order = [cv[n] for n in INNERS]
MONOTONE_UP = all(b >= a for a, b in zip(order, order[1:]))
print()
print(f"CV in grid order: {[round(v, 6) for v in order]}")
print(f"monotone rising in N_INNER: {MONOTONE_UP}  "
      "(the train/serve mismatch argument predicts yes)")

blocked = None
if not (LEAK_OK and CLEAN):
    blocked = "a leak check failed"
elif not ENCODER_MATCH:
    blocked = "the encoder does not match 13"
elif not DETERMINISTIC:
    blocked = "the configuration is not reproducible"
elif not INNER_LIVE:
    blocked = "N_INNER is not reaching the encoder, so these are not five arms"
elif not VAL_HELD:
    blocked = "N_INNER moved the validation encoding, which it must never do"
elif not SMOKE and not ALIGNED:
    blocked = "fold alignment failed"
elif not SMOKE and not REPRODUCED:
    blocked = f"the N_INNER={BASE_INNER} arm missed row 38 by {repro:+.2e}"

print()
if blocked:
    print(f"VERDICT: blocked, {blocked}")
elif SMOKE:
    print("SMOKE: no verdict, subsampled rows cannot resolve differences this small.")
else:
    best = max(rows, key=lambda r: r[1])
    n, m, sdv, w, _ = best
    sig = w >= 4 and sdv > 0 and m > 2 * sdv
    if sig and m >= GATE_FLOOR:
        print(f"VERDICT: carry to a second seed. N_INNER={n} clears the bar at {m:+.6f},")
        print("  and one seed does not make an improvement in this repo (rows 26, 38).")
    elif sig:
        print(f"VERDICT: parity. N_INNER={n} is paired-significant at {m:+.6f} but under")
        print(f"  the {GATE_FLOOR:.5f} floor. Logged as parity, not as an improvement.")
    else:
        print("VERDICT: null. The curve is flat and 13's N_INNER=5 stands, which is the")
        print("  sixth point on the rule in NOTES.md and the second knob to return")
        print("  nothing on this representation.")


In [ ]:
pre = "SMOKE_" if SMOKE else ""
for n in INNERS:
    np.save(OUT / f"{pre}xgb_inner{n}_oof.npy", oof[n])
    np.save(OUT / f"{pre}xgb_inner{n}_test.npy", test_pred[n])
print(f"wrote {pre}xgb_inner<n>_oof.npy and _test.npy for {INNERS}")
print()
print("ledger lines, one per arm:")
for n in INNERS:
    print(f"  xgb_inner{str(n):<4}  cv_mean {cv[n]:.6f}  cv_std {sd[n]:.6f}")
print()
print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}, "
      f"encoder {'matches 13' if ENCODER_MATCH else 'DIFFERS'}, "
      f"determinism {'OK' if DETERMINISTIC else 'FAILED'}, "
      f"inner live {'yes' if INNER_LIVE else 'NO'}, "
      f"valid held {'yes' if VAL_HELD else 'NO'}")
